# Phase 2: Factor Construction


## Factors (FF 2015 definitions, Indian adaptation: FY ends 31 March)

| Factor | Sort variable | Definition (formation year t) |
|---|---|---|
| MKT | — | VW return of all stocks − Rf (VW uses ME at end of t−1 month) |
| SMB | Size | ME at end of June t |
| HML | B/M | BE (FY ending Mar t) / ME (end of Mar t) |
| RMW | OP | (Operating profit − Interest) / BE, FY ending Mar t |
| CMA | Inv | (TA_Mar t − TA_Mar t−1) / TA_Mar t−1 |

OP follows FF: revenue − COGS − SG&A − interest, divided by book equity
(other income excluded). Inv uses the lagged total-assets denominator.

## Timing
- Portfolios formed at **end of June t**, held **July t → June t+1**.
- Lag: SEBI LODR requires audited annual results within 60 days of FY end,
  so March-t accounting data is public by June t (FF use ≥6 months for US 10-K lags).

## Breakpoints
- Size: median June ME of all stocks with valid June ME (one breakpoint per year,
  shared by all three sorts).
- Characteristics: 30th/70th percentiles of **Big stocks only**, applied to all stocks.
  This follows FF's international factor convention (big-stock breakpoints as the
  NYSE-breakpoint analog).

## Eligibility (per sort)
- All: common equity, valid June ME > 0
- HML: BE > 0, valid March ME
- RMW: BE > 0, valid OP
- CMA: valid TA for both March t and March t−1

## Weighting
- Value-weighted with ME at end of previous month.
- A portfolio-month with fewer than MIN_STOCKS stocks is set to NaN
  (documented deviation; FF impose no minimum because the US universe is large).

## Known limitations
- Universe is current Nifty 500 constituents, so survivorship and selection look-ahead are present.
- About 245 stocks per year have data (not 500). "Small" here means mid-cap, so SMB ≈ mid minus large.
- Returns are price returns unless dividends were added in Phase 1, so MKT is understated.
- No delisting returns are available for NSE.
- Independent sorts (FF convention) CAN produce thin cells; they do not prevent them.

In [1]:
# ==========================================================
# STEP 0: Imports + Paths
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

# Methodology constants
BP_LOW = 0.30  # characteristic low breakpoint
BP_HIGH = 0.70  # characteristic high breakpoint
FORM_MONTH = 6  # June snapshot
MIN_STOCKS = 5  # min stocks per portfolio-month (documented deviation)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

In [2]:
# ==========================================================
# STEP 1: Load Phase 1 Outputs
# ==========================================================

final = pd.read_csv(PROCESSED_DIR / "monthly_panel.csv", parse_dates=["date"])

chars = pd.read_csv(PROCESSED_DIR / "characteristics_annual.csv")

rf = pd.read_csv(PROCESSED_DIR / "rf_monthly.csv")

print("final panel:", final.shape)
print(final.columns.tolist())

print("\nchars:", chars.shape)
print(chars.columns.tolist())

final panel: (69084, 14)
['symbol', 'date', 'ret', 'close', 'shares', 'me', 'me_lag', 'shares_source', 'trade_days', 'tt_days', 'Rf_monthly', 'excess_ret', 'is_financial', 'formation_date']

chars: (5598, 29)
['symbol', 'fy_end', 'year', 'quarter', 'sales', 'operating_profit', 'other_income', 'interest_expense', 'depreciation', 'net_income', 'total_assets', 'book_value', 'basis', 'operating_profitability', 'is_financial', 'months_since_prev_fy', 'prev_basis', 'check_me', 'available_date', 'prev_total_assets', 'asset_growth', 'me_fy_end', 'bm', 'formation_date', 'me_formation', 'shares_source', 'months_traded', 'tv_median_12m', 'trade_days_12m']


In [3]:
final.columns

Index(['symbol', 'date', 'ret', 'close', 'shares', 'me', 'me_lag',
       'shares_source', 'trade_days', 'tt_days', 'Rf_monthly', 'excess_ret',
       'is_financial', 'formation_date'],
      dtype='object')

In [4]:
rf.head()

,date,Rf_monthly
0,2011-11-30,NaN
1,2011-12-31,0.007391
2,2012-01-31,0.007128
3,2012-02-29,0.007179
4,2012-03-31,0.007432


In [5]:
rf["Rf_monthly"].describe(percentiles=[0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

count    176.000000
mean       0.005342
std        0.001389
min        0.002546
1%         0.002654
10%        0.002928
25%        0.004520
50%        0.005398
75%        0.006196
90%        0.007144
99%        0.007896
max        0.009445
Name: Rf_monthly, dtype: float64

In [6]:
# ==========================================================
# STEP 2: Data Checks on Phase 1 Outputs
# ==========================================================

# 2.1 header and date parsing

# CSV reload turns dates into text -> convert back
for col in ["fy_end", "available_date", "formation_date"]:
    chars[col] = pd.to_datetime(chars[col])

final["formation_date"] = pd.to_datetime(final["formation_date"])

In [7]:
# 2.2 keeping only those columns which are used in phase 2

cols_needed = [
    "symbol",
    "date",
    "ret",
    "me",
    "me_lag",  # weight for month t = ME at end of t-1
    "Rf_monthly",
    "is_financial",
    "formation_date",  # links each month to its June formation
]

final = final[cols_needed].copy()

In [8]:
# 2.3 basic cleaning

# inf can appear from divisions in Phase 1 (e.g., zero prior assets)
final = final.replace([np.inf, -np.inf], np.nan)
chars = chars.replace([np.inf, -np.inf], np.nan)

# Sort by stock, then time (needed for the me_lag check)
final = final.sort_values(["symbol", "date"]).reset_index(drop=True)

In [9]:
# 2.4 uniqueness/deduplication

dup_final = final.duplicated(["symbol", "date"]).sum()
dup_chars = chars.duplicated(["symbol", "formation_date"]).sum()

print("Duplicate stock-months :", dup_final)  # must be 0
print("Duplicate stock-years  :", dup_chars)  # must be 0

Duplicate stock-months : 0
Duplicate stock-years  : 0


In [10]:
# 2.5 missing values overview

print(final.isnull().sum())

print()
char_cols = [
    "me_formation",
    "book_value",
    "bm",
    "operating_profitability",
    "asset_growth",
]
print(chars[char_cols].isnull().sum())

symbol               0
date                 0
ret                462
me                3017
me_lag            3250
Rf_monthly        4125
is_financial         0
formation_date       0
dtype: int64

me_formation               1192
book_value                    0
bm                         1228
operating_profitability     795
asset_growth                663
dtype: int64


In [11]:
# 2.6 risk-free rate checks

print(final["Rf_monthly"].describe())

# Months with no Rf -> MKT cannot be computed there
missing_rf = final.loc[final["Rf_monthly"].isna(), "date"]
print(missing_rf.drop_duplicates().sort_values())

# Suspiciously low values (Indian T-bill never ~0%)
print(final.nsmallest(5, "Rf_monthly")[["date", "Rf_monthly"]])

count    64959.000000
mean         0.005218
std          0.001347
min          0.002546
25%          0.004485
50%          0.005342
75%          0.005824
max          0.009445
Name: Rf_monthly, dtype: float64
113   2011-01-31
114   2011-02-28
115   2011-03-31
116   2011-04-30
117   2011-05-31
118   2011-06-30
119   2011-07-31
120   2011-08-31
121   2011-09-30
122   2011-10-31
123   2011-11-30
82    2026-08-31
83    2026-09-30
Name: date, dtype: datetime64[ns]
          date  Rf_monthly
14  2020-12-31    0.002546
232 2020-12-31    0.002546
328 2020-12-31    0.002546
517 2020-12-31    0.002546
706 2020-12-31    0.002546


In [12]:
# 2.7 verifying phase 1's ME lag

me_prev = final.groupby("symbol")["me"].shift(1)

diff = (me_prev - final["me_lag"]).abs()
print(diff.describe())  # max should be ~0

print("me_lag missing:", final["me_lag"].isna().sum())

count    65834.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
dtype: float64
me_lag missing: 3250


In [13]:
# 2.8 drop missing returns and check extremes

final = final.dropna(subset=["ret"])

# -50% / +100% in a month: real event or missed split/bonus
extreme = final[(final["ret"] < -0.5) | (final["ret"] > 1.0)]

print("Extreme returns:", len(extreme))
print(extreme[["date", "symbol", "ret"]].sort_values("ret"))

Extreme returns: 249
            date      symbol       ret
17530 2024-04-30       CUPID -0.955285
62765 2011-06-30       TITAN -0.951383
60609 2025-10-31  TATAINVEST -0.922656
3932  2026-02-28    ANGELONE -0.908221
10401 2017-04-30         BLS -0.907098
...          ...         ...       ...
64201 2020-10-31        TTML  1.525424
46477 2013-07-31  NEULANDLAB  1.545728
21675 2022-12-31        FACT  1.558068
51689 2021-02-28  POONAWALLA  1.611296
56401 2016-07-31  SHRIPISTON  1.652501

[249 rows x 3 columns]


In [14]:
# Detect split/bonus months

final["me_ret"] = final["me"] / final["me_lag"] - 1
final["split_factor"] = (1 + final["me_ret"]) / (1 + final["ret"])

suspect = (final["ret"] - final["me_ret"]).abs() > 0.25

# Price fell much more than market cap
price_drop = final["ret"] < -0.25

# Factor matches a standard split/bonus ratio (within 2%)
std_ratios = [1.5, 2, 3, 4, 5, 10, 20]
is_clean = pd.Series(False, index=final.index)
for r in std_ratios:
    is_clean = is_clean | np.isclose(final["split_factor"], r, rtol=0.02)

grp_a = suspect & price_drop & is_clean  # clean split -> fix
grp_b = suspect & price_drop & ~is_clean  # messy split -> manual
grp_c = suspect & ~price_drop  # share jump  -> ret is fine

print("A clean splits :", grp_a.sum())
print("B messy splits :", grp_b.sum())
print("C share jumps  :", grp_c.sum())

A clean splits : 77
B messy splits : 12
C share jumps  : 155


In [15]:
# Fix group A (clean splits) — safe to re-run

if "ret_raw" not in final.columns:
    final["ret_raw"] = final["ret"]  # untouched Phase 1 return

final["ret"] = final["ret_raw"]  # reset before any fix

show = ["date", "symbol", "ret", "me_ret", "split_factor"]
print(final.loc[grp_a, show].round(3))

final.loc[grp_a, "ret"] = final.loc[grp_a, "me_ret"]
print("Returns corrected:", grp_a.sum())

            date      symbol    ret  me_ret  split_factor
41    2023-03-31      360ONE -0.750   0.000         4.002
2729  2021-10-31       AFFLE -0.804  -0.018         4.999
3660  2025-03-31  ANANDRATHI -0.522  -0.044         2.000
3675  2026-06-30  ANANDRATHI -0.427   0.146         2.000
4319  2020-12-31   APLAPOLLO -0.739   0.306         4.998
...          ...         ...    ...     ...           ...
65698 2019-07-31         UPL -0.365  -0.047         1.501
67602 2024-12-31       WIPRO -0.478   0.045         2.001
67921 2020-03-31     YESBANK -0.351   2.190         4.916
68281 2018-09-30  ZENSARTECH -0.831  -0.155         5.004
69071 2025-09-30   ZYDUSWELL -0.774   0.132         5.000

[77 rows x 5 columns]
Returns corrected: 77


In [16]:
# Manual fix for group B (ratios from NSE corp actions)

print(final.loc[grp_b, show].round(3))  # the 12 rows to look up

b_ratios = {
    ("ANGELONE", "2026-02-28"): 10,  # verify on NSE
    # ('SYMBOL', 'YYYY-MM-DD'): ratio,
}

for (sym, dt), ratio in b_ratios.items():
    row = (final["symbol"] == sym) & (final["date"] == dt)
    final.loc[row, "ret"] = (1 + final.loc[row, "ret_raw"]) * ratio - 1
    print(sym, dt, "->", final.loc[row, "ret"].round(3).tolist())

            date      symbol    ret  me_ret  split_factor
2999  2026-01-31        AIIL -0.839  -0.211         4.895
3932  2026-02-28    ANGELONE -0.908  -0.495         5.502
7661  2025-06-30  BAJFINANCE -0.898  -0.438         5.508
23493 2018-07-31       GICRE -0.496  -0.027         1.930
36883 2022-04-30    JUBLFOOD -0.793  -0.032         4.675
44507 2022-11-30       MSUMI -0.298  -0.017         1.400
47804 2022-11-30       NYKAA -0.851  -0.103         6.002
53741 2017-09-30    RELIANCE -0.510  -0.057         1.926
66118 2021-06-30         VBL -0.279   0.040         1.442
66130 2022-06-30         VBL -0.252   0.099         1.469
66142 2023-06-30         VBL -0.526  -0.274         1.533
66157 2024-09-30         VBL -0.596   0.010         2.500
ANGELONE 2026-02-28 -> [-0.082]


In [17]:
# Classify group C (ret is fine; me / me_lag may not be)

split_syms = final.loc[grp_a | grp_b, "symbol"].unique()

c_late = grp_c & final["symbol"].isin(split_syms)  # likely late share update
c_real = grp_c & ~final["symbol"].isin(split_syms)  # likely real issuance

print("C late share updates :", c_late.sum())
print("C real issuance      :", c_real.sum())

print(final.loc[c_late, show].sort_values(["symbol", "date"]).round(3))

final = final.drop(columns=["me_ret", "split_factor"])

C late share updates : 32
C real issuance      : 123
            date      symbol    ret  me_ret  split_factor
3934  2026-04-30    ANGELONE  0.357   1.472         1.822
4329  2021-10-31   APLAPOLLO -0.026   0.944         1.996
7662  2025-07-31  BAJFINANCE -0.059   0.710         1.818
11560 2025-06-30         BSE  0.036   2.046         2.940
11944 2019-12-31       CANBK -0.026   0.333         1.368
11948 2020-04-30       CANBK -0.029   0.368         1.409
15879 2026-05-31     COFORGE  0.189   0.522         1.280
19547 2022-10-31      ECLERX -0.074   0.389         1.500
25494 2021-11-30        GPIL -0.202   1.918         3.658
27379 2026-01-31     HDFCAMC -0.059   0.251         1.328
28726 2017-08-31   HINDPETRO  0.274   0.529         1.200
30783 2018-11-30         IEX  0.024   9.213         9.975
31172 2017-12-31         IGL  0.028   0.712         1.665
31258 2025-02-28         IGL -0.061   0.878         2.000
33830 2022-01-31         IRB  0.118   0.921         1.718
33844 2023-03-31   

77 clean split/bonus months patched with ME-implied returns; 12 patched manually with NSE corporate-action ratios. X stock-months show lagged share-count updates (affects me/me_lag weights only); pending Phase 1 fix.

In [18]:
# 2.9 annual FY data check

# Mostly month 3 (March FY); a few Dec/Sep MNC year-ends are fine
print(chars["fy_end"].dt.month.value_counts())

# Must be one value or NaN; several values = quarterly, not annual
print(chars["quarter"].value_counts(dropna=False))

fy_end
3     5433
12     142
9       13
6        9
2        1
Name: count, dtype: int64
quarter
FY    5598
Name: count, dtype: int64


| FY ends in | Rows | Who these usually are |
|---|---|---|
| March | 5,433 (97%) | Standard Indian April–March year |
| December | 142 | MNC subsidiaries following their parent's calendar year |
| September | 13 | e.g., Siemens-type parents |
| June | 9 | Rare, or a company in transition |
| February | 1 | Odd, probably a one-off transition period or a data error |

Lags will be different for different FY endings, for ex: 6 for FY ending in Dec.

In [19]:
# 2.10 look-ahead guard

# Accounting data must be public before the June formation date
late = chars["available_date"] > chars["formation_date"]
print("Look-ahead rows:", late.sum())  # must be 0

# Months between FY end and formation
year_gap = chars["formation_date"].dt.year - chars["fy_end"].dt.year
month_gap = chars["formation_date"].dt.month - chars["fy_end"].dt.month
lag_months = year_gap * 12 + month_gap

print(lag_months.value_counts().sort_index())

Look-ahead rows: 0
3     5433
4        1
6      142
9       13
12       9
Name: count, dtype: int64


In [20]:
# Valid: FY ended 3 to 14 months before June formation

stale = (lag_months < 3) | (lag_months > 14)
print("Stale or too-recent rows:", stale.sum())

print(chars.loc[stale, ["symbol", "fy_end", "formation_date"]])

chars = chars[~stale].copy()

Stale or too-recent rows: 0
Empty DataFrame
Columns: [symbol, fy_end, formation_date]
Index: []


In [21]:
# 2.11 OP formula check

# FF: OP = (Revenue - COGS - SG&A - Interest) / Book Equity
op_ff = chars["operating_profit"] - chars["interest_expense"]
op_ff = op_ff / chars["book_value"]

gap = (op_ff - chars["operating_profitability"]).abs()
print(gap.describe())  # max should be ~0

count    4.803000e+03
mean     3.300032e-17
std      6.455751e-17
min      0.000000e+00
25%      0.000000e+00
50%      2.775558e-17
75%      5.551115e-17
max      3.552714e-15
dtype: float64


In [22]:
# 2.12 B/M check

# FF: B/M = BE / ME at fiscal year end
bm_ff = chars["book_value"] / chars["me_fy_end"]

gap_bm = (bm_ff - chars["bm"]).abs()
print(gap_bm.describe())  # max should be ~0

count    4.370000e+03
mean     3.870458e-17
std      7.113207e-17
min      0.000000e+00
25%      0.000000e+00
50%      2.775558e-17
75%      5.551115e-17
max      1.776357e-15
dtype: float64


In [23]:
# 2.13 Inv consistency

period = chars["months_since_prev_fy"]

first_obs = chars["prev_basis"].isna()  # no prior FY
switch = ~first_obs & (chars["basis"] != chars["prev_basis"])
data_gap = period.isin([24, 36])  # missing year(s)
transition = period.notna() & (period % 12 != 0)  # 9, 15 months

print("First observations :", first_obs.sum())
print("Basis switches     :", switch.sum())
print("Data gaps (24/36)  :", data_gap.sum())
print("Transition (9/15)  :", transition.sum())

First observations : 489
Basis switches     : 140
Data gaps (24/36)  : 15
Transition (9/15)  : 21


In [24]:
# Inv: needs a clean 12-month change on the same basis
bad_inv = switch | data_gap | transition
had_inv = chars["asset_growth"].notna()

print("Asset growth actually lost:", (bad_inv & had_inv).sum())
chars.loc[bad_inv, "asset_growth"] = np.nan

# OP: only transition periods are non-annual
had_op = chars["operating_profitability"].notna()
print("OP actually lost:", (transition & had_op).sum())
chars.loc[transition, "operating_profitability"] = np.nan

Asset growth actually lost: 0
OP actually lost: 21


In [25]:
print(chars["basis"].value_counts(dropna=False))
print(chars.loc[switch, ["basis", "prev_basis"]].value_counts())

basis
consolidated    4859
standalone       739
Name: count, dtype: int64
basis         prev_basis  
consolidated  standalone      111
standalone    consolidated     29
Name: count, dtype: int64


**FF timing convention (Indian adaptation):**

- Portfolios formed at **end of June t** (`formation_date` = 30 June t)
- Accounting data: **FY ending March t**. Audited annual results are public
  by end of May under SEBI LODR's 60-day rule, so there is no look-ahead.
  (FF use a ≥6-month lag for US 10-K filings.)
- Size: ME at end of June t. B/M: BE / ME at FY end (March t)
- Portfolios held **July t → June t+1**, rebalanced annually
- Monthly value weights use ME at end of the previous month (`me_lag`)

In [26]:
# ==========================================================
# STEP 3: FF Annual Formation Dataset
# ==========================================================

**What this step does:** prepares one row per stock per June formation date,
holding everything the 2×3 sorts need.

**FF timing (Indian adaptation):**

| Item | Value used | FF original (US) |
|---|---|---|
| Formation date | 30 June t | 30 June t |
| Size | ME at end of June t | ME at end of June t |
| B/M | BE (FY ≤ March t) / ME end of March t | BE (FY in t−1) / ME end of Dec t−1 |
| OP, Inv | FY ending ≤ March t (12-month periods only) | FY ending in t−1 |
| Holding period | July t → June t+1 | July t → June t+1 |

**Source:** `characteristics_annual.csv` from Phase 1, already checked in Step 2 for
look-ahead (2.10), formulas (2.11–2.12) and period consistency (2.13).

**Link to monthly returns:** every month in `final` carries a `formation_date`;
Jul t – Jun t+1 → 30 June t.

In [27]:
# 3.1 Check Phase 1's month -> formation_date mapping

year = final["date"].dt.year
month = final["date"].dt.month

# FF rule: Jul-Dec -> same year, Jan-Jun -> previous year
expected_year = np.where(month >= 7, year, year - 1)

wrong_year = final["formation_date"].dt.year != expected_year
wrong_month = final["formation_date"].dt.month != 6

print("Rows with wrong formation year :", wrong_year.sum())  # must be 0
print("Rows with non-June formation   :", wrong_month.sum())  # must be 0

# Visual check around one boundary (June -> July 2019)
boundary = final[final["date"].between("2019-05-31", "2019-08-31")]
print(boundary[["date", "formation_date"]].drop_duplicates())

Rows with wrong formation year : 0
Rows with non-June formation   : 0
          date formation_date
213 2019-05-31     2018-06-30
214 2019-06-30     2018-06-30
215 2019-07-31     2019-06-30
216 2019-08-31     2019-06-30


Results for respective months look correct

In [28]:
# 3.2 Formation snapshot: one row per stock per June

keep_cols = [
    "symbol",
    "formation_date",
    "me_formation",  # size: ME at end of June t
    "book_value",  # for the positive-BE filter
    "bm",  # HML sort variable
    "operating_profitability",  # RMW sort variable
    "asset_growth",  # CMA sort variable
    "is_financial",
    "check_me",  # Phase 1 ME-quality flag
    "trade_days_12m",  # 0 = no trading -> stale June pric
]

formation = chars[keep_cols].copy()

formation = formation.rename(
    columns={
        "me_formation": "me_june",
        "operating_profitability": "op",
        "asset_growth": "inv",
    }
)

print(formation.shape)
print(formation.head())

(5598, 10)
   symbol formation_date  me_june  book_value  bm        op       inv  \
0     IOB     2002-06-30      NaN      1099.0 NaN       NaN       NaN   
1  COLPAL     2006-06-30      NaN       281.0 NaN  0.637011       NaN   
2  COLPAL     2007-06-30      NaN       278.0 NaN  0.712230  0.106583   
3  RADICO     2007-06-30      NaN       185.0 NaN  0.227027       NaN   
4  COLPAL     2008-06-30      NaN       168.0 NaN  1.488095  0.032578   

   is_financial  check_me  trade_days_12m  
0          True     False             NaN  
1         False     False             NaN  
2         False     False             NaN  
3         False     False             NaN  
4         False     False             NaN  


In [29]:
# 3.3 Does chars' June ME match the panel's June ME?

june = final[final["date"].dt.month == 6][["symbol", "date", "me"]]
june = june.rename(columns={"date": "formation_date", "me": "me_june_panel"})

keys = ["symbol", "formation_date"]
check = formation.merge(june, on=keys, how="left")

ratio = check["me_june"] / check["me_june_panel"]
print(ratio.describe())

off = (ratio - 1).abs() > 0.01
print("Rows where June ME differs by > 1%:", off.sum())
print(check.loc[off, keys + ["me_june", "me_june_panel"]])

count    4400.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64
Rows where June ME differs by > 1%: 0
Empty DataFrame
Columns: [symbol, formation_date, me_june, me_june_panel]
Index: []


In [30]:
# 3.4 Uniqueness and coverage per year

dups = formation.duplicated(["symbol", "formation_date"]).sum()
print("Duplicate stock-years:", dups)  # must be 0

has = pd.DataFrame(
    {
        "formation_date": formation["formation_date"],
        "stocks": 1,
        "me_june": formation["me_june"] > 0,
        "bm": formation["bm"] > 0,
        "op": formation["op"].notna(),
        "inv": formation["inv"].notna(),
    }
)

coverage = has.groupby("formation_date").sum()
print(coverage)

Duplicate stock-years: 0
                stocks  me_june   bm   op  inv
formation_date                                
2002-06-30           1        0    0    0    0
2006-06-30           1        0    0    1    0
2007-06-30           2        0    0    2    1
2008-06-30           2        0    0    2    1
2009-06-30           4        0    0    3    2
2010-06-30           5        0    0    4    4
2011-06-30           7        3    1    6    4
2012-06-30           6        3    3    6    5
2013-06-30           6        3    3    6    5
2014-06-30           7        1    1    6    3
2015-06-30         399      288  281  336    6
2016-06-30         416      293  289  347  383
2017-06-30         422      307  298  360  395
2018-06-30         438      305  301  370  398
2019-06-30         452      363  352  385  414
2020-06-30         476      373  365  405  436
2021-06-30         488      391  378  417  460
2022-06-30         491      415  398  420  479
2023-06-30         492      423  41

In [31]:
# 3.5 Share of monthly returns that can enter factor portfolios

keys = ["symbol", "formation_date"]

linked = final[keys + ["date"]].merge(
    formation[keys], on=keys, how="left", indicator=True
)

linked["in_formation"] = linked["_merge"] == "both"

by_year = linked.groupby(linked["date"].dt.year)["in_formation"].mean()
print(by_year.round(2))

date
2011    0.01
2012    0.01
2013    0.01
2014    0.01
2015    0.50
2016    0.99
2017    1.00
2018    1.00
2019    1.00
2020    1.00
2021    1.00
2022    1.00
2023    1.00
2024    1.00
2025    1.00
2026    1.00
Name: in_formation, dtype: float64


In [32]:
early = formation[formation["formation_date"].dt.year < 2015]
print(early[["symbol", "formation_date", "bm", "op", "inv"]])

       symbol formation_date        bm        op       inv
0         IOB     2002-06-30       NaN       NaN       NaN
1      COLPAL     2006-06-30       NaN  0.637011       NaN
2      COLPAL     2007-06-30       NaN  0.712230  0.106583
3      RADICO     2007-06-30       NaN  0.227027       NaN
4      COLPAL     2008-06-30       NaN  1.488095  0.032578
5   TATAELXSI     2008-06-30       NaN  0.613445       NaN
6      COLPAL     2009-06-30       NaN  1.319444  0.096022
7    LLOYDSME     2009-06-30       NaN  0.524390       NaN
8   TATAELXSI     2009-06-30       NaN  0.513158 -0.065637
9        TTML     2009-06-30       NaN       NaN       NaN
10     COLPAL     2010-06-30       NaN  1.305136  0.107635
11   LLOYDSME     2010-06-30       NaN  0.250000 -0.032680
12  PATANJALI     2010-06-30       NaN  0.090113       NaN
13  TATAELXSI     2010-06-30       NaN  0.377143  0.223140
14       TTML     2010-06-30       NaN       NaN  0.194486
15      GVT&D     2011-06-30       NaN  0.356929       N

40 stock-years spread across 2011–2014 is only about 10 stocks per year. A 2×3 sort needs six portfolios, so you’d get 1–2 stocks per cell, and those “factor returns” would be single-stock noise, not factors. Removing these stocks.

In [33]:
SAMPLE_START = "2015-06-30"

before = len(formation)
formation = formation[formation["formation_date"] >= SAMPLE_START].copy()

print("Pre-2015 formation rows removed:", before - len(formation))

Pre-2015 formation rows removed: 41


In [34]:
# 3.6 Financials: kept in all sorts except RMW

is_fin = formation["is_financial"].fillna(False).astype(bool)

# How much data do financials have for each sort variable?
print(formation.loc[is_fin, ["bm", "op", "inv"]].notna().sum())

# OP is not comparable for banks/NBFCs -> drop them from RMW only
formation.loc[is_fin, "op"] = np.nan

print("Financial stock-years kept   :", is_fin.sum())
print("Excluded from RMW sort only  :", is_fin.sum())

bm     835
op     419
inv    987
dtype: int64
Financial stock-years kept   : 1120
Excluded from RMW sort only  : 1120


**Treatment of financial firms**

- **Kept** in MKT, SMB, HML and CMA, following FF (2015), which includes all common stocks.
- **Excluded from RMW only.** For non-financial firms, OP = (Revenue − COGS − SG&A − Interest) / BE,
  where interest is a financing cost deducted after operations. For banks and NBFCs, interest is the
  core cost of doing business (their "raw material") and there is no COGS, so the same formula
  is not economically comparable.
- **Robustness check:** actors will also be rebuilt without financials and compared.

In [35]:
# 3.7 What are these Phase 1 columns? (inspect, don't filter yet)

flag_cols = ["check_me", "months_traded", "tv_median_12m", "trade_days_12m"]

print(chars[flag_cols].describe(include="all"))
print(chars["check_me"].value_counts(dropna=False).head())

       check_me  months_traded  tv_median_12m  trade_days_12m
count      5598    4683.000000   4.502000e+03     4502.000000
unique        2            NaN            NaN             NaN
top       False            NaN            NaN             NaN
freq       5586            NaN            NaN             NaN
mean        NaN     101.821695   2.191565e+10      245.546868
std         NaN      50.153541   4.028776e+10       12.367688
min         NaN       1.000000   0.000000e+00        0.000000
25%         NaN      66.000000   2.122107e+09      246.000000
50%         NaN     102.000000   8.363103e+09      247.000000
75%         NaN     144.000000   2.438341e+10      249.000000
max         NaN     186.000000   6.705395e+11      251.000000
check_me
False    5586
True       12
Name: count, dtype: int64


In [36]:
# 3.7b FF requirement: a valid June ME

keys = ["symbol", "formation_date"]

no_me = ~(formation["me_june"] > 0)  # NaN or zero ME
no_trade = formation["trade_days_12m"] == 0  # stale June price
invalid = no_me | no_trade

print("Removed (no valid June ME):", invalid.sum())
formation = formation[~invalid].copy()

# Rows flagged by Phase 1 but with a valid-looking ME -> review
flag_only = formation["check_me"].astype(bool)

review = formation[flag_only].merge(june, on=keys, how="left")
review["ratio"] = review["me_june"] / review["me_june_panel"]

print(review[keys + ["me_june", "me_june_panel", "ratio"]].round(3))

Removed (no valid June ME): 1161
     symbol formation_date     me_june  me_june_panel  ratio
0      BBOX     2019-06-30     361.005        361.005    1.0
1  TATACOMM     2021-06-30   36686.469      36686.469    1.0
2    INDIGO     2021-06-30   66612.682      66612.682    1.0
3      LICI     2022-06-30  426255.237     426255.237    1.0
4    PFOCUS     2023-06-30    3477.093       3477.093    1.0


In [37]:
print("Flagged rows to review:", flag_only.sum())  # expect 4 (or 5 with LICI 2022)

Flagged rows to review: 5


In [38]:
# Drop only if chars' ME disagrees with the panel by more than 1%
mismatch = (review["ratio"] - 1).abs() > 0.01
drop_keys = review.loc[mismatch, keys]

print("Flagged rows dropped:", len(drop_keys))
print("Flagged rows kept   :", len(review) - len(drop_keys))

merged = formation.merge(drop_keys, on=keys, how="left", indicator=True)
formation = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

Flagged rows dropped: 0
Flagged rows kept   : 5


In [39]:
missing_td = formation["trade_days_12m"].isna()
print(missing_td.groupby(formation["formation_date"].dt.year).sum())

formation_date
2015     5
2016     8
2017    12
2018    18
2019    11
2020    10
2021    18
2022    22
2023     9
2024    17
2025    21
2026    16
Name: trade_days_12m, dtype: int64


In [40]:
# 3.7c Keep or drop the flagged rows

# Drop only if chars' ME disagrees with the panel by more than 1%
review["mismatch"] = (review["ratio"] - 1).abs() > 0.01

print(review[keys + ["me_june", "me_june_panel", "ratio", "mismatch"]].round(3))

drop_keys = review.loc[review["mismatch"], keys]

print("Flagged rows dropped:", len(drop_keys))
print("Flagged rows kept   :", len(review) - len(drop_keys))

merged = formation.merge(drop_keys, on=keys, how="left", indicator=True)
formation = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

print("Formation rows left:", len(formation))

     symbol formation_date     me_june  me_june_panel  ratio  mismatch
0      BBOX     2019-06-30     361.005        361.005    1.0     False
1  TATACOMM     2021-06-30   36686.469      36686.469    1.0     False
2    INDIGO     2021-06-30   66612.682      66612.682    1.0     False
3      LICI     2022-06-30  426255.237     426255.237    1.0     False
4    PFOCUS     2023-06-30    3477.093       3477.093    1.0     False
Flagged rows dropped: 0
Flagged rows kept   : 5
Formation rows left: 4396


In [41]:
# ==========================================================
# STEP 4: Size Breakpoint and Eligibility
# ==========================================================

**Size breakpoint (FF):** one median of June ME per formation year, computed on
all stocks with a valid June ME (FF use the NYSE median). The same Small/Big label
is used in all three 2×3 sorts.

**Eligibility per sort (FF):**

| Sort | Required | Note |
|---|---|---|
| HML | BE > 0, valid B/M | Negative-BE firms excluded (FF) |
| RMW | BE > 0, valid OP | BE is OP's denominator; financials excluded (Step 3.6) |
| CMA | Valid asset growth | 12-month change, same statement basis (Step 2.13) |

No winsorizing of B/M, OP or Inv: sorts are rank-based, and FF don't trim.

In [42]:
# 4.1 Size: median June ME per formation year

size_bp = formation.groupby("formation_date")["me_june"].median()

formation["size_bp"] = formation["formation_date"].map(size_bp)

is_small = formation["me_june"] <= formation["size_bp"]
formation["SIZE"] = np.where(is_small, "S", "B")

print(formation.groupby(["formation_date", "SIZE"]).size().unstack())

SIZE              B    S
formation_date          
2015-06-30      144  144
2016-06-30      146  147
2017-06-30      153  154
2018-06-30      152  153
2019-06-30      181  182
2020-06-30      186  187
2021-06-30      195  196
2022-06-30      207  208
2023-06-30      211  212
2024-06-30      221  221
2025-06-30      233  234
2026-06-30      164  165


In [43]:
# 4.2 Who can enter each 2x3 sort

pos_be = formation["book_value"] > 0

hml_ok = pos_be & (formation["bm"] > 0)
rmw_ok = pos_be & formation["op"].notna()
cma_ok = formation["inv"].notna()

eligible_hml = formation[hml_ok].copy()
eligible_rmw = formation[rmw_ok].copy()
eligible_cma = formation[cma_ok].copy()

print("HML eligible stock-years:", len(eligible_hml))
print("RMW eligible stock-years:", len(eligible_rmw))
print("CMA eligible stock-years:", len(eligible_cma))

HML eligible stock-years: 4267
RMW eligible stock-years: 3491
CMA eligible stock-years: 4038


In [44]:
# CMA needs two years of total assets -> first sort June 2016

CMA_START = "2016-06-30"
eligible_cma = eligible_cma[eligible_cma["formation_date"] >= CMA_START].copy()

print("CMA eligible stock-years:", len(eligible_cma))

CMA eligible stock-years: 4036


In [45]:
LAST_FORMATION = '2025-06-30'
for df in [eligible_hml, eligible_rmw, eligible_cma]:
    df.drop(df[df['formation_date'] > LAST_FORMATION].index, inplace=True)

In [46]:
# 4.3 Eligible stocks per year (total and Big)


def count_by_year(df):
    total = df.groupby("formation_date").size()
    big = df[df["SIZE"] == "B"].groupby("formation_date").size()
    return pd.DataFrame({"total": total, "big": big})


counts = pd.concat(
    {
        "HML": count_by_year(eligible_hml),
        "RMW": count_by_year(eligible_rmw),
        "CMA": count_by_year(eligible_cma),
    },
    axis=1,
)

print(counts)

                 HML        RMW         CMA       
               total  big total  big  total    big
formation_date                                    
2015-06-30       281  142   239  113    NaN    NaN
2016-06-30       285  143   237  113  276.0  136.0
2017-06-30       295  146   255  119  300.0  149.0
2018-06-30       298  152   245  111  298.0  147.0
2019-06-30       352  178   288  136  355.0  181.0
2020-06-30       364  184   296  141  371.0  185.0
2021-06-30       377  191   309  150  384.0  193.0
2022-06-30       396  195   324  159  411.0  207.0
2023-06-30       417  207   331  157  416.0  205.0
2024-06-30       429  217   348  166  437.0  219.0
2025-06-30       459  230   372  172  462.0  231.0


In [47]:
# Why does the June 2026 formation have fewer stocks?
n_chars = chars.groupby("formation_date").size()
print(n_chars.tail(3))

late_fy = chars[chars["formation_date"] == "2026-06-30"]
print(late_fy["fy_end"].value_counts())

formation_date
2024-06-30    493
2025-06-30    495
2026-06-30    495
dtype: int64
fy_end
2026-03-31    486
2025-12-31      8
2025-09-30      1
Name: count, dtype: int64


**Sample per factor**
- HML, RMW: sortable from June 2015 (≥ 237 stocks, ≥ 111 Big per year)
- CMA: from June 2016 (asset growth needs two years of total assets)
- SMB and the joint FF5 sample: from **July 2016** (~120 months to June 2026)
- June 2026 formation: partial coverage (329 vs 467 stocks), excluded until FY2026 data is complete

In [48]:
# 4.4 Value ranges (check only, no trimming)

pct = [0.01, 0.30, 0.50, 0.70, 0.99]

print(eligible_hml["bm"].describe(percentiles=pct).round(3))
print(eligible_rmw["op"].describe(percentiles=pct).round(3))
print(eligible_cma["inv"].describe(percentiles=pct).round(3))

count    3953.000
mean        0.535
std         0.956
min         0.002
1%          0.023
30%         0.172
50%         0.295
70%         0.505
99%         3.543
max        34.883
Name: bm, dtype: float64
count    3244.000
mean        0.253
std         0.830
min       -19.423
1%         -0.217
30%         0.160
50%         0.223
70%         0.289
99%         1.023
max        33.400
Name: op, dtype: float64
count    3710.000
mean        0.598
std        27.121
min        -0.716
1%         -0.213
30%         0.056
50%         0.111
70%         0.175
99%         1.196
max      1652.000
Name: inv, dtype: float64


In [49]:
# ==========================================================
# STEP 5: Market Factor (MKT)
# ==========================================================

$$
MKT_t = R_{m,t} - R_{f,t}, \qquad
R_{m,t} = \frac{\sum_i ME_{i,t-1} \, r_{i,t}}{\sum_i ME_{i,t-1}}
$$

| Item | Choice | FF convention |
|---|---|---|
| Universe | All stocks in the panel with a return and lagged ME, **financials included** | All CRSP common stocks |
| Weights | ME at end of **previous** month (`me_lag`) | Same |
| Rf | Monthly 91-day T-bill rate (`Rf_monthly`) | One-month T-bill |
| Returns | Split/bonus-corrected (Step 2.8) | CRSP adjusted returns |

MKT uses the **full panel**, not just stocks with accounting data, so it runs from 2011.
Limitation: universe = current Nifty 500 constituents (survivorship).

In [50]:
# 5.1 Value-weighted market return (weights = last month's ME)

mkt_data = final.dropna(subset=["ret", "me_lag"]).copy()
mkt_data = mkt_data[mkt_data["me_lag"] > 0]

# Each stock's return x its weight (before dividing by total ME)
mkt_data["w_ret"] = mkt_data["ret"] * mkt_data["me_lag"]

mkt = (
    mkt_data.groupby("date")
    .agg(
        w_ret=("w_ret", "sum"),
        total_me=("me_lag", "sum"),
        n_stocks=("symbol", "count"),
    )
    .reset_index()
)

mkt["market_return"] = mkt["w_ret"] / mkt["total_me"]

In [51]:
# 5.2 Risk-free rate and market excess return

# Rf must be a single value per month (same for all stocks)
rf_values = final.groupby("date")["Rf_monthly"].nunique()
print("Months with more than one Rf value:", (rf_values > 1).sum())  # must be 0

rf_by_date = final.groupby("date")["Rf_monthly"].first()

mkt["Rf"] = mkt["date"].map(rf_by_date)
mkt["MKT"] = mkt["market_return"] - mkt["Rf"]

print("Months without Rf:", mkt["Rf"].isna().sum())
print(mkt.loc[mkt["Rf"].isna(), "date"].dt.strftime("%Y-%m").tolist())

Months with more than one Rf value: 0
Months without Rf: 12
['2011-02', '2011-03', '2011-04', '2011-05', '2011-06', '2011-07', '2011-08', '2011-09', '2011-10', '2011-11', '2026-08', '2026-09']


Only edge months don't have Rf value (2011 and 2026) and that is fair.

In [52]:
# 5.3 Does MKT look like the Indian market?

# (a) Breadth: stocks per month, averaged by year
print(mkt.groupby(mkt["date"].dt.year)["n_stocks"].mean().round(0))


date
2011    284.0
2012    290.0
2013    290.0
2014    286.0
2015    293.0
2016    297.0
2017    306.0
2018    312.0
2019    363.0
2020    371.0
2021    377.0
2022    409.0
2023    424.0
2024    440.0
2025    468.0
2026    387.0
Name: n_stocks, dtype: float64


In [53]:
# ----------------------------------------------------------
# 5.3(b) MKT premium (FF view) + wealth growth (investor view)
# ----------------------------------------------------------
w = mkt[mkt['date'] >= '2016-07-31'].dropna(subset=['Rf'])
n = len(w)
years = n / 12

# --- FF view: arithmetic mean of monthly MKT, with t-stat ---
mkt_mean = w['MKT'].mean()
mkt_sd = w['MKT'].std()
t_stat = mkt_mean / (mkt_sd / np.sqrt(n))

# --- Investor view: growth of Rs 100 ---
growth_m = (1 + w['market_return']).prod()
growth_f = (1 + w['Rf']).prod()

cagr_m = growth_m ** (1 / years) - 1
cagr_f = growth_f ** (1 / years) - 1
geo_excess = (1 + cagr_m) / (1 + cagr_f) - 1

# --- Consistency check: arithmetic ~ CAGR + variance / 2 ---
arith_m = w['market_return'].mean() * 12
drag = (w['market_return'].std() * np.sqrt(12)) ** 2 / 2

print(w['date'].min().date(), '->', w['date'].max().date(), '|', n, 'months')
print()
print('FF VIEW (used in the model)')
print(f'  MKT mean monthly     : {mkt_mean * 100:.2f}%')
print(f'  MKT mean annualised  : {mkt_mean * 12 * 100:.2f}%')
print(f'  t-stat               : {t_stat:.2f}')
print()
print('INVESTOR VIEW (sanity check)')
print(f'  Rs 100 in market     -> Rs {100 * growth_m:.0f}  (CAGR {cagr_m * 100:.2f}%)')
print(f'  Rs 100 in T-bills    -> Rs {100 * growth_f:.0f}  (CAGR {cagr_f * 100:.2f}%)')
print(f'  Geometric excess     : {geo_excess * 100:.2f}% per year')
print()
print('CHECK')
print(f'  Arithmetic market    : {arith_m * 100:.2f}%')
print(f'  CAGR + drag          : {(cagr_m + drag) * 100:.2f}%   (should be close)')

2016-07-31 -> 2026-07-31 | 121 months

FF VIEW (used in the model)
  MKT mean monthly     : 0.46%
  MKT mean annualised  : 5.54%
  t-stat               : 1.00

INVESTOR VIEW (sanity check)
  Rs 100 in market     -> Rs 262  (CAGR 10.03%)
  Rs 100 in T-bills    -> Rs 176  (CAGR 5.75%)
  Geometric excess     : 4.05% per year

CHECK
  Arithmetic market    : 11.14%
  CAGR + drag          : 11.55%   (should be close)


In [54]:
# (c) Best and worst months -> should match known market events
show = ["date", "market_return", "n_stocks"]
print(mkt.nsmallest(5, "market_return")[show].round(3))
print(mkt.nlargest(5, "market_return")[show].round(3))

          date  market_return  n_stocks
109 2020-03-31         -0.219       371
91  2018-09-30         -0.123       311
46  2014-12-31         -0.120       290
0   2011-02-28         -0.112       280
181 2026-03-31         -0.111       489
          date  market_return  n_stocks
11  2012-01-31          0.159       290
110 2020-04-30          0.137       369
39  2014-05-31          0.131       290
182 2026-04-30          0.115       393
19  2012-09-30          0.110       291


In [55]:
# 5.3(d) Which stock-months drag the market return most?

month_total = mkt_data.groupby('date')['me_lag'].transform('sum')
mkt_data['weight'] = mkt_data['me_lag'] / month_total
mkt_data['contrib'] = mkt_data['weight'] * mkt_data['ret']

show = ['date', 'symbol', 'ret', 'weight', 'contrib']
print(mkt_data.nsmallest(15, 'contrib')[show].round(4))

            date      symbol     ret  weight  contrib
55516 2014-11-30        SBIN -0.8811  0.1483  -0.1307
48654 2011-02-28        ONGC -0.7705  0.1022  -0.0787
29889 2014-12-31   ICICIBANK -0.7987  0.0790  -0.0631
7556  2016-09-30  BAJFINANCE -0.9041  0.0582  -0.0526
9073  2015-09-30         BEL -0.6630  0.0723  -0.0479
32676 2014-12-31        INFY -0.5476  0.0773  -0.0423
36631 2017-01-31    JSWSTEEL -0.8782  0.0456  -0.0401
27394 2011-07-31    HDFCBANK -0.8065  0.0495  -0.0399
5426  2013-07-31  ASIANPAINT -0.8905  0.0424  -0.0378
29027 2011-03-31    HINDZINC -0.8911  0.0408  -0.0363
9091  2017-03-31         BEL -0.8960  0.0368  -0.0330
10027 2011-10-31        BHEL -0.8062  0.0399  -0.0322
62765 2011-06-30       TITAN -0.9514  0.0299  -0.0285
55474 2011-05-31        SBIN -0.1701  0.1613  -0.0274
6715  2014-07-31    AXISBANK -0.7959  0.0334  -0.0266


In [56]:
# 5.4 Is any single stock dominating the market return?

top_idx = mkt_data.groupby("date")["weight"].idxmax()
top = mkt_data.loc[top_idx, ["date", "symbol", "weight"]]

print(top["weight"].describe().round(3))
print(top.nlargest(5, "weight").round(3))

count    188.000
mean       0.081
std        0.042
min        0.036
25%        0.050
50%        0.063
75%        0.110
max        0.168
Name: weight, dtype: float64
            date symbol  weight
55487 2012-06-30   SBIN   0.168
55484 2012-03-31   SBIN   0.167
55488 2012-07-31   SBIN   0.167
55494 2013-01-31   SBIN   0.166
55486 2012-05-31   SBIN   0.165


Highest weight of less than 20% is fine from the Indian context.

In [57]:
# 5.5 Clean market-factor table for later steps

mkt_factor = mkt[['date', 'market_return', 'Rf', 'MKT']].copy()

print(mkt_factor[60:65])
print(mkt_factor.isnull().sum())

mkt_factor.to_csv(PROCESSED_DIR / 'mkt_factor.csv', index=False)

         date  market_return        Rf       MKT
60 2016-02-29      -0.069915  0.004832 -0.074747
61 2016-03-31       0.099493  0.004595  0.094897
62 2016-04-30       0.020220  0.006049  0.014171
63 2016-05-31       0.025747  0.005760  0.019987
64 2016-06-30       0.029385  0.005590  0.023794
date              0
market_return     0
Rf               12
MKT              12
dtype: int64


### Step 5 Summary: Market Factor (MKT)

**Construction:** MKTₜ = R_m,t − R_f,t, built monthly with simple returns. R_m is the
value-weighted return of all panel stocks (financials included), weighted by last
month's ME (`me_lag`). Rf is the monthly 91-day T-bill rate. Returns are split/bonus-corrected.

**Coverage:** Dec 2011 to Jul 2026 (176 months); Rf missing only at the 2011 and 2026 edges.

**Results (Jul 2016 – Jul 2026, 121 months)**

| Measure | Value |
|---|---|
| MKT mean, annualised (t-stat) | 1.00 |
| ₹100 in market → ₹262 | CAGR 10.03% |
| ₹100 in T-bills → ₹176 | CAGR 5.75% |

**Checks:** worst month __March '20__ (-21.9%), best month __January '12__ (15.9%); largest single-stock
weight 16.8% of SBIN (< 20%).

**Limitations:** current-constituent universe (survivorship); full-cap rather than
free-float weights; price returns only.

**Output:** `mkt_factor.csv` (`date`, `market_return`, `Rf`, `MKT`), used in Step 11 and Phase 3.

In [58]:
# ==========================================================
# STEP 6: Helper Function for 2×3 Sorts
# ==========================================================

This function constructs:
- **HML** portfolios
- **RMW** portfolios
- **CMA** portfolios

using:

- Small / Big
- Low / Neutral / High

In [59]:
# methodology: NSE-adapted FF5 using median size and 30/70 characteristic breakpoints

In [60]:
# Generic 2x3 Sort Function

def assign_2x3_portfolios(df, variable, low_label='L', high_label='H'):

    df = df.copy()
    df = df.dropna(subset=[variable, 'me_june'])

    # valid positive market capitalization
    df = df[df['me_june'] > 0].copy()

    # CHARACTERISTIC BREAKPOINTS
    # ------------------------------------------------------
    # 30th and 70th percentiles ONLY among BIG stocks within each formation year.
    # These breakpoints are then applied to BOTH Small and Big stocks.
    # ------------------------------------------------------
    big = df[df['SIZE'] == 'B'].copy()

    low_breakpoint = (
        big.groupby('formation_date')[variable]
        .quantile(0.30)
        .rename('low_breakpoint')
    )

    high_breakpoint = (
        big.groupby('formation_date')[variable]
        .quantile(0.70)
        .rename('high_breakpoint')
    )

    # Attach Big-stock breakpoints to ALL stocks
    df = df.merge(
        pd.concat([low_breakpoint, high_breakpoint], axis=1),
        on='formation_date',
        how='left'
    )

    # CHARACTERISTIC BUCKETS
    df['CHAR'] = 'N'
    df.loc[df[variable] <= df['low_breakpoint'], 'CHAR'] = low_label
    df.loc[df[variable] >= df['high_breakpoint'], 'CHAR'] = high_label

    # FINAL 2x3 PORTFOLIO
    df['PORT'] = df['SIZE'] + df['CHAR']

    return df

Assigns each stock to one of six portfolios: **Size** (S/B) × **Characteristic** (Low/Neutral/High).

- Size: from Step 4.1 (one median June ME per year, shared by all sorts)
- Characteristic breakpoints: 30th / 70th percentiles of **Big stocks only**,
  applied to all stocks (FF international convention, the analog of NYSE breakpoints)
- Used for HML (B/M), RMW (OP) and CMA (Inv)

In [61]:
# ==========================================================
# STEP 7: HML Construction
# ==========================================================

$$
HML = \tfrac{1}{2}(SH + BH) - \tfrac{1}{2}(SL + BL)
$$

- **H** = Value (high B/M), **L** = Growth (low B/M)
- Eligible stocks: Step 4.2 (`eligible_hml`: BE > 0, valid B/M)
- Portfolios formed end of June t, held July t → June t+1
- Monthly returns value-weighted with **last month's ME** (`me_lag`)
- A portfolio-month with fewer than `MIN_STOCKS` stocks is set to NaN

In [62]:
# 7.1 Assign each eligible stock to a 2x3 portfolio

# eligible_hml comes from Step 4.2 -> not redefined here
hml_assign = assign_2x3_portfolios(
    eligible_hml,
    'bm',
    low_label='L',
    high_label='H'
)

print(hml_assign['PORT'].value_counts().sort_index())

PORT
BH    598
BL    598
BN    789
SH    959
SL    244
SN    765
Name: count, dtype: int64


In [63]:
# 7.2 Each stock keeps its portfolio from July t to June t+1

keys = ['symbol', 'formation_date']

hml_monthly = final.merge(
    hml_assign[keys + ['PORT']],
    on=keys,
    how='inner'          # keep only stocks that have a portfolio
)

print(hml_monthly['PORT'].value_counts().sort_index())

PORT
BH     7167
BL     7149
BN     9455
SH    11398
SL     2890
SN     9145
Name: count, dtype: int64


In [64]:
# 7.3 Monthly VW returns (weights = last month's ME)

MIN_STOCKS = 5    # thinner portfolio-months -> NaN (documented deviation)

def vw_portfolio_returns(monthly):
    df = monthly.dropna(subset=['ret', 'me_lag'])
    df = df[df['me_lag'] > 0].copy()

    df['w_ret'] = df['ret'] * df['me_lag']

    ports = df.groupby(['date', 'PORT']).agg(
        w_ret=('w_ret', 'sum'),
        total_me=('me_lag', 'sum'),
        n_stocks=('symbol', 'count'),
    ).reset_index()

    ports['ret'] = ports['w_ret'] / ports['total_me']
    ports.loc[ports['n_stocks'] < MIN_STOCKS, 'ret'] = np.nan
    return ports

hml_ports = vw_portfolio_returns(hml_monthly)
print(hml_ports.head())

        date PORT          w_ret      total_me  n_stocks       ret
0 2015-07-31   BH  -26761.403731  2.394688e+06        43 -0.011175
1 2015-07-31   BL    9987.459862  3.409781e+06        43  0.002929
2 2015-07-31   BN  129825.121725  4.390881e+06        56  0.029567
3 2015-07-31   SH   19195.116836  2.758700e+05        84  0.069580
4 2015-07-31   SL    4619.365055  3.244592e+04         8  0.142371


In [65]:
# 7.4 Six portfolios side by side -> HML

hml_pivot = hml_ports.pivot(index='date', columns='PORT', values='ret')

# All six columns must exist (SN, BN needed later for SMB)
hml_pivot = hml_pivot.reindex(columns=['SL', 'SN', 'SH', 'BL', 'BN', 'BH'])

value = (hml_pivot['SH'] + hml_pivot['BH']) / 2
growth = (hml_pivot['SL'] + hml_pivot['BL']) / 2
hml_pivot['HML'] = value - growth

print(hml_pivot.head())

hml_factor = hml_pivot[['HML']].reset_index()

print('HML factor'.center('-'*10))
print(hml_factor.head())

PORT              SL        SN        SH        BL        BN        BH  \
date                                                                     
2015-07-31  0.142371  0.066706  0.069580  0.002929  0.029567 -0.011175   
2015-08-31 -0.081671 -0.021273 -0.076108 -0.084071 -0.041759 -0.104522   
2015-09-30 -0.037802 -0.001317  0.024040 -0.187427 -0.010551 -0.003078   
2015-10-31  0.012142  0.045714  0.051966 -0.054538  0.010337  0.034440   
2015-11-30  0.037835  0.004002  0.036366  0.013740 -0.019885  0.013036   

PORT             HML  
date                  
2015-07-31 -0.043448  
2015-08-31 -0.007444  
2015-09-30  0.123096  
2015-10-31  0.064401  
2015-11-30 -0.001086  


TypeError: 'str' object cannot be interpreted as an integer

In [66]:
# 7.5 Are portfolios well populated? Does HML look sensible?

# (a) Stocks per portfolio per formation year
per_year = hml_assign.groupby(['formation_date', 'PORT']).size()
print(per_year.unstack(fill_value=0))

# (b) Months where HML is missing
print('HML months:', hml_factor['HML'].notna().sum())
print('HML missing:', hml_factor['HML'].isna().sum())

# (c) Premium: mean, t-stat
h = hml_factor['HML'].dropna()
t_stat = h.mean() / (h.std() / np.sqrt(len(h)))
print(f'HML mean annualised: {h.mean() * 12 * 100:.2f}%  |  t-stat: {t_stat:.2f}')

PORT            BH  BL  BN   SH  SL   SN
formation_date                          
2015-06-30      43  43  56   84   8   47
2016-06-30      43  43  57   78  16   48
2017-06-30      44  44  58   74  25   50
2018-06-30      46  46  60   63  18   65
2019-06-30      54  54  70   88  14   72
2020-06-30      55  55  74  112   8   60
2021-06-30      58  58  75  106  15   65
2022-06-30      59  59  77  108  21   72
2023-06-30      62  62  83   91  31   88
2024-06-30      65  65  87   66  45  101
2025-06-30      69  69  92   89  43   97
HML months: 132
HML missing: 0
HML mean annualised: 10.21%  |  t-stat: 2.30


In [67]:
# 7.6 How many company-months came under which PORT how many times?

hml_monthly.head()

,symbol,date,ret,me,me_lag,Rf_monthly,is_financial,formation_date,ret_raw,PORT
0,360ONE,2020-07-31,0.047468,9525.300346,9093.646931,0.002759,True,2020-06-30,0.047468,SN
1,360ONE,2020-08-31,-0.043681,9109.222054,9525.300346,0.002681,True,2020-06-30,-0.043681,SN
2,360ONE,2020-09-30,-0.060332,8559.642707,9109.222054,0.002695,True,2020-06-30,-0.060332,SN
3,360ONE,2020-10-31,-0.043462,8187.619764,8559.642707,0.002729,True,2020-06-30,-0.043462,SN
4,360ONE,2020-11-30,0.104027,9039.356502,8187.619764,0.002711,True,2020-06-30,0.104027,SN


In [68]:
port_symbols = (hml_monthly
                .groupby(['PORT', 'symbol'])
                .size()
                .reset_index(name='months'))

port_symbols = port_symbols.sort_values(['PORT', 'months'],
                                        ascending=[True, False])

In [69]:
with pd.option_context('display.max_rows', None):
    display(port_symbols)

,PORT,symbol,months
11,BH,BAJAJHLDNG,132
13,BH,BANKBARODA,132
14,BH,BANKINDIA,132
19,BH,CANBK,132
39,BH,HINDALCO,132
52,BH,IOC,132
77,BH,NHPC,132
80,BH,NMDC,132
81,BH,NTPC,132
88,BH,PNB,132


In [70]:
# ==========================================================
# STEP 8: RMW Construction
# ==========================================================

$$
RMW = \tfrac{1}{2}(SR + BR) - \tfrac{1}{2}(SW + BW)
$$

- **R** = Robust (high OP), **W** = Weak (low OP)
- OP = (Operating profit − Interest) / BE, FY ending ≤ March t
- Eligible stocks: Step 4.2 (`eligible_rmw`: BE > 0, valid OP); **financials excluded** (Step 3.6)
- Portfolios formed end of June t, held July t → June t+1
- Monthly returns value-weighted with `me_lag`; portfolio-months below `MIN_STOCKS` set to NaN

In [71]:
# 8.1 Assign each eligible stock to a 2x3 portfolio

# eligible_rmw comes from Step 4.2 -> not redefined here
rmw_assign = assign_2x3_portfolios(
    eligible_rmw,
    'op',
    low_label='W',
    high_label='R'
)

print(rmw_assign['PORT'].value_counts().sort_index())

# Financials must not be in RMW (Step 3.6)
fin_in_rmw = rmw_assign['is_financial'].fillna(False).astype(bool).sum()
print('Financial stock-years in RMW:', fin_in_rmw)       # must be 0

PORT
BN    609
BR    464
BW    464
SN    638
SR    256
SW    813
Name: count, dtype: int64
Financial stock-years in RMW: 0


In [72]:
# 8.2 Each stock keeps its portfolio from July t to June t+1

keys = ['symbol', 'formation_date']

rmw_monthly = final.merge(
    rmw_assign[keys + ['PORT']],
    on=keys,
    how='inner'
)

print(rmw_monthly['PORT'].value_counts().sort_index())

PORT
BN    7289
BR    5560
BW    5546
SN    7626
SR    3044
SW    9642
Name: count, dtype: int64


In [73]:
# 8.3 Monthly VW returns (same function as Step 7.3)

rmw_ports = vw_portfolio_returns(rmw_monthly)
print(rmw_ports.head())

        date PORT          w_ret      total_me  n_stocks       ret
0 2015-07-31   BN   53669.321413  2.927390e+06        45  0.018334
1 2015-07-31   BR  -54407.575896  2.951667e+06        34 -0.018433
2 2015-07-31   BW  203854.121068  2.430392e+06        34  0.083877
3 2015-07-31   SN   20349.566715  1.955857e+05        45  0.104044
4 2015-07-31   SR    6153.888390  4.298789e+04        13  0.143154


In [74]:
# 8.4 Six portfolios side by side -> RMW

rmw_pivot = rmw_ports.pivot(index='date', columns='PORT', values='ret')

# All six columns must exist (SN, BN needed later for SMB)
rmw_pivot = rmw_pivot.reindex(columns=['SW', 'SN', 'SR', 'BW', 'BN', 'BR'])

robust = (rmw_pivot['SR'] + rmw_pivot['BR']) / 2
weak = (rmw_pivot['SW'] + rmw_pivot['BW']) / 2
rmw_pivot['RMW'] = robust - weak

rmw_factor = rmw_pivot[['RMW']].reset_index()
print(rmw_factor.tail())

PORT       date       RMW
127  2026-02-28 -0.023303
128  2026-03-31  0.018805
129  2026-04-30 -0.072115
130  2026-05-31  0.012650
131  2026-06-30 -0.047935


In [75]:
# 8.5 Are portfolios well populated? Does RMW look sensible?

# (a) Stocks per portfolio per formation year
per_year = rmw_assign.groupby(['formation_date', 'PORT']).size()
print(per_year.unstack(fill_value=0))

# (b) Months where RMW is missing
print('RMW months:', rmw_factor['RMW'].notna().sum())
print('RMW missing:', rmw_factor['RMW'].isna().sum())

# (c) Premium: mean, t-stat
r = rmw_factor['RMW'].dropna()
t_stat = r.mean() / (r.std() / np.sqrt(len(r)))
print(f'RMW mean annualised: {r.mean() * 12 * 100:.2f}%  |  t-stat: {t_stat:.2f}')

PORT            BN  BR  BW  SN  SR  SW
formation_date                        
2015-06-30      45  34  34  45  13  68
2016-06-30      45  34  34  53  19  52
2017-06-30      47  36  36  45  19  72
2018-06-30      43  34  34  43  21  70
2019-06-30      54  41  41  59  20  73
2020-06-30      55  43  43  56  26  73
2021-06-30      60  45  45  58  18  83
2022-06-30      63  48  48  55  31  79
2023-06-30      63  47  47  74  25  75
2024-06-30      66  50  50  66  33  83
2025-06-30      68  52  52  84  31  85
RMW months: 132
RMW missing: 0
RMW mean annualised: -1.87%  |  t-stat: -0.52


In [76]:
rmw_symbols = (rmw_monthly
               .groupby(['PORT', 'symbol'])
               .size()
               .reset_index(name='months')
               .sort_values(['PORT', 'months'], ascending=[True, False]))

print(rmw_symbols.to_string())

    PORT      symbol  months
61    BN     HAVELLS     132
24    BN  BALKRISIND     108
110   BN        ONGC     108
137   BN       TECHM     108
156   BN       WIPRO     108
45    BN     DRREDDY      96
70    BN         IGL      96
89    BN          LT      96
105   BN        NMDC      96
112   BN    PETRONET      96
150   BN         UPL      96
160   BN   ZYDUSLIFE      96
12    BN       ALKEM      84
19    BN      ASTRAL      84
23    BN  BAJAJ-AUTO      84
26    BN         BEL      84
34    BN       CIPLA      84
40    BN       DABUR      84
58    BN    GODREJCP      84
59    BN      GRASIM      84
93    BN         M&M      84
95    BN      MARUTI      84
99    BN     MPHASIS      84
122   BN    SHREECEM      84
128   BN   SUNPHARMA      84
44    BN       DMART      82
7     BN  ADANIPORTS      72
28    BN  BHARATFORG      72
50    BN    EXIDEIND      72
55    BN       GLAXO      72
77    BN         ITC      72
106   BN        NTPC      72
114   BN       PIIND      72
115   BN     P

In [77]:
# ==========================================================
# STEP 9: CMA FACTOR (FULLY SELF-CONTAINED)
# ==========================================================

$$
CMA = \tfrac{1}{2}(SC + BC) - \tfrac{1}{2}(SA + BA)
$$

- **C** = Conservative (low asset growth), **A** = Aggressive (high asset growth)
- Inv = (TA_Mar t − TA_Mar t−1) / TA_Mar t−1; 12-month change on the same statement basis (Step 2.13)
- Eligible stocks: Step 4.2 (`eligible_cma`); first formation **June 2016** (Step 4.2b),
  since asset growth needs two years of total assets
- Financials included (asset growth is comparable across firms)
- Portfolios formed end of June t, held July t → June t+1
- Monthly returns value-weighted with `me_lag`; portfolio-months below `MIN_STOCKS` set to NaN

In [78]:
# 9.1 Assign each eligible stock to a 2x3 portfolio

# eligible_cma comes from Step 4.2 / 4.2b -> not redefined here
cma_assign = assign_2x3_portfolios(
    eligible_cma,
    'inv',
    low_label='C',       # low asset growth  = Conservative
    high_label='A'       # high asset growth = Aggressive
)

print(cma_assign['PORT'].value_counts().sort_index())
print('First formation:', cma_assign['formation_date'].min().date())   # expect 2016-06-30

PORT
BA    559
BC    559
BN    735
SA    543
SC    704
SN    610
Name: count, dtype: int64
First formation: 2016-06-30


In [79]:
# 9.2 Each stock keeps its portfolio from July t to June t+1

keys = ['symbol', 'formation_date']

cma_monthly = final.merge(
    cma_assign[keys + ['PORT']],
    on=keys,
    how='inner'
)

print(cma_monthly['PORT'].value_counts().sort_index())

PORT
BA    6692
BC    6687
BN    8811
SA    6466
SC    8321
SN    7272
Name: count, dtype: int64


In [80]:
# 9.3 Monthly VW returns (same function as Step 7.3)

cma_ports = vw_portfolio_returns(cma_monthly)
print(cma_ports.head())

        date PORT          w_ret      total_me  n_stocks       ret
0 2016-07-31   BA  237413.001509  2.923608e+06        41  0.081205
1 2016-07-31   BC  199481.106938  2.433395e+06        41  0.081976
2 2016-07-31   BN  102814.452633  2.914368e+06        54  0.035278
3 2016-07-31   SA   11477.701282  1.341204e+05        31  0.085578
4 2016-07-31   SC   13077.094603  2.271845e+05        63  0.057562


In [81]:
# 9.4 Six portfolios side by side -> CMA

cma_pivot = cma_ports.pivot(index='date', columns='PORT', values='ret')

# All six columns must exist (SN, BN needed later for SMB)
cma_pivot = cma_pivot.reindex(columns=['SC', 'SN', 'SA', 'BC', 'BN', 'BA'])

conservative = (cma_pivot['SC'] + cma_pivot['BC']) / 2
aggressive = (cma_pivot['SA'] + cma_pivot['BA']) / 2
cma_pivot['CMA'] = conservative - aggressive

cma_factor = cma_pivot[['CMA']].reset_index()
print(cma_factor.head())

PORT       date       CMA
0    2016-07-31 -0.013623
1    2016-08-31 -0.012844
2    2016-09-30  0.072281
3    2016-10-31 -0.021774
4    2016-11-30  0.029774


In [82]:
# 9.5 Are portfolios well populated? Does CMA look sensible?

# (a) Stocks per portfolio per formation year
per_year = cma_assign.groupby(['formation_date', 'PORT']).size()
print(per_year.unstack(fill_value=0))

# (b) Months where CMA is missing
print('CMA months:', cma_factor['CMA'].notna().sum())
print('CMA missing:', cma_factor['CMA'].isna().sum())

# (c) Premium: mean, t-stat
c = cma_factor['CMA'].dropna()
t_stat = c.mean() / (c.std() / np.sqrt(len(c)))
print(f'CMA mean annualised: {c.mean() * 12 * 100:.2f}%  |  t-stat: {t_stat:.2f}')


PORT            BA  BC  BN  SA  SC  SN
formation_date                        
2016-06-30      41  41  54  31  63  46
2017-06-30      45  45  59  33  66  52
2018-06-30      44  44  59  41  59  51
2019-06-30      55  55  71  47  71  56
2020-06-30      56  56  73  41  88  57
2021-06-30      58  58  77  39  87  65
2022-06-30      62  62  83  71  54  79
2023-06-30      62  62  81  75  71  65
2024-06-30      66  66  87  76  70  72
2025-06-30      70  70  91  89  75  67
CMA months: 120
CMA missing: 0
CMA mean annualised: 1.02%  |  t-stat: 0.36


In [83]:
# (d) Asset growth distribution per year (old describe cell)
pct = [0.30, 0.50, 0.70]
print(cma_assign.groupby('formation_date')['inv'].describe(percentiles=pct).round(3))

# (e) CMA vs HML correlation (FF 2015: strongly positive in the US)
both = hml_factor.merge(cma_factor, on='date').dropna()
print('Corr(HML, CMA):', round(both['HML'].corr(both['CMA']), 2))

                count   mean     std    min    30%    50%    70%       max
formation_date                                                            
2016-06-30      276.0  6.081  99.433 -0.688  0.017  0.082  0.149  1652.000
2017-06-30      300.0  0.124   0.193 -0.377  0.036  0.097  0.162     1.323
2018-06-30      298.0  0.174   0.419 -0.716  0.061  0.122  0.194     5.767
2019-06-30      355.0  0.159   0.393 -0.402  0.052  0.101  0.175     5.585
2020-06-30      371.0  0.098   0.203 -0.636  0.024  0.076  0.124     1.544
2021-06-30      384.0  0.136   0.250 -0.329  0.032  0.088  0.154     2.411
2022-06-30      411.0  0.181   0.252 -0.511  0.077  0.131  0.199     2.255
2023-06-30      416.0  0.156   0.222 -0.282  0.065  0.115  0.182     2.513
2024-06-30      437.0  0.172   0.196 -0.259  0.081  0.132  0.196     1.301
2025-06-30      462.0  0.197   0.298 -0.633  0.089  0.134  0.196     4.370
Corr(HML, CMA): 0.31


In [84]:
cma_symbols = (cma_monthly
               .groupby(['PORT', 'symbol'])
               .size()
               .reset_index(name='months')
               .sort_values(['PORT', 'months'], ascending=[True, False]))

print(cma_symbols.to_string())

     PORT      symbol  months
28     BA  BAJAJFINSV     108
31     BA  BAJFINANCE     108
225    BA    UNOMINDA      96
23     BA      AUBANK      84
217    BA       TITAN      84
221    BA    TVSMOTOR      84
104    BA         IGL      72
108    BA      INDIGO      72
136    BA         LTM      72
156    BA      NAUKRI      72
18     BA    ASHOKLEY      60
33     BA  BANDHANBNK      60
41     BA      BIOCON      60
42     BA   BRITANNIA      60
67     BA   EICHERMOT      60
83     BA  GODREJPROP      60
137    BA        LTTS      60
153    BA  MUTHOOTFIN      60
176    BA       PIIND      60
6      BA  ADANIGREEN      57
4      BA  ADANIENSOL      48
24     BA  AUROPHARMA      48
30     BA  BAJAJHLDNG      48
48     BA    CHOLAFIN      48
58     BA    CROMPTON      48
64     BA       DIXON      48
73     BA  FEDERALBNK      48
87     BA     HAVELLS      48
90     BA    HDFCBANK      48
109    BA  INDUSINDBK      48
120    BA   JSWENERGY      48
123    BA    JUBLFOOD      48
169    BA 

In [85]:
# ==========================================================
# STEP 10: SMB Construction (Canonical FF5)
# ==========================================================

$$
SMB = \tfrac{1}{3}\left(SMB_{B/M} + SMB_{OP} + SMB_{Inv}\right)
$$

Each component = average of the 3 Small portfolios − average of the 3 Big portfolios
from that 2×3 sort (canonical FF5).

- SMB is NaN if any component is missing, so it starts in **July 2016** (when CMA starts)
- Size breakpoint: one June median for all sorts (Step 4.1)

In [86]:
# 10.1 Size spread from each 2x3 sort

def size_spread(pivot, small_cols, big_cols):
    small = pivot[small_cols].mean(axis=1, skipna=False) # skipna=False: if one portfolio is missing, the spread is NaN
    big = pivot[big_cols].mean(axis=1, skipna=False)
    return small - big

smb_hml = size_spread(hml_pivot, ['SL', 'SN', 'SH'], ['BL', 'BN', 'BH'])
smb_rmw = size_spread(rmw_pivot, ['SW', 'SN', 'SR'], ['BW', 'BN', 'BR'])
smb_cma = size_spread(cma_pivot, ['SC', 'SN', 'SA'], ['BC', 'BN', 'BA'])

In [87]:
# 10.2 SMB = average of the three size spreads

smb = pd.concat([smb_hml, smb_rmw, smb_cma], axis=1)
smb.columns = ['SMB_HML', 'SMB_RMW', 'SMB_CMA']

smb['SMB'] = smb.mean(axis=1, skipna=False)

smb_factor = smb[['SMB']].reset_index()
print(smb_factor.dropna().head())

         date       SMB
12 2016-07-31  0.005808
13 2016-08-31  0.013244
14 2016-09-30  0.061957
15 2016-10-31  0.084601
16 2016-11-30 -0.034959


In [88]:
# 10.3 Coverage, premium, consistency

# (a) Which component is missing, and when?
print(smb[['SMB_HML', 'SMB_RMW', 'SMB_CMA']].isna().sum())
print('SMB first month:', smb['SMB'].first_valid_index().date())    # expect 2016-07-31

# (b) Premium: mean, t-stat
s = smb['SMB'].dropna()
t_stat = s.mean() / (s.std() / np.sqrt(len(s)))
print(f'SMB mean annualised: {s.mean() * 12 * 100:.2f}%  |  t-stat: {t_stat:.2f}')

# (c) The three spreads should move together
print(smb[['SMB_HML', 'SMB_RMW', 'SMB_CMA']].corr().round(2))

SMB_HML     0
SMB_RMW     0
SMB_CMA    12
dtype: int64
SMB first month: 2016-07-31
SMB mean annualised: 15.58%  |  t-stat: 4.02
         SMB_HML  SMB_RMW  SMB_CMA
SMB_HML     1.00     0.93     0.96
SMB_RMW     0.93     1.00     0.93
SMB_CMA     0.96     0.93     1.00


In [89]:
# SMB_CMA has 12 months missing in 2015-2016 only, nothing in between 2016 and 2026

In [90]:
# 10.4 Missing months per portfolio (lists renamed: no clash)

hml_cols = ['SL', 'SN', 'SH', 'BL', 'BN', 'BH']
rmw_cols = ['SW', 'SN', 'SR', 'BW', 'BN', 'BR']
cma_cols = ['SC', 'SN', 'SA', 'BC', 'BN', 'BA']

for name, pivot, cols in [('HML', hml_pivot, hml_cols),
                          ('RMW', rmw_pivot, rmw_cols),
                          ('CMA', cma_pivot, cma_cols)]:
    print(f'\n{name} portfolios, missing months:')
    print(pivot[cols].isna().sum().to_string())


HML portfolios, missing months:
PORT
SL    0
SN    0
SH    0
BL    0
BN    0
BH    0

RMW portfolios, missing months:
PORT
SW    0
SN    0
SR    0
BW    0
BN    0
BR    0

CMA portfolios, missing months:
PORT
SC    0
SN    0
SA    0
BC    0
BN    0
BA    0


In [91]:
# ==========================================================
# STEP 11: Final FF5 Factor Dataset
# ==========================================================

One row per month: `MKT, SMB, HML, RMW, CMA, Rf` (decimals, e.g. 0.012 = 1.2%).
Common sample = months where **all five factors** exist (from July 2016).
Saved to `data/processed/ff5_factors.csv` for Phase 3.

In [92]:
# 11.1 Put all factors side by side, aligned on date

ff5 = pd.concat([
    mkt_factor.set_index('date')[['MKT', 'Rf']],
    smb_factor.set_index('date')['SMB'],
    hml_factor.set_index('date')['HML'],
    rmw_factor.set_index('date')['RMW'],
    cma_factor.set_index('date')['CMA'],
], axis=1)

ff5 = ff5[['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'Rf']]   # FF column order

print(len(ff5))
print(ff5.isnull().sum())

188
MKT    12
SMB    68
HML    56
RMW    56
CMA    68
Rf     12
dtype: int64


In [93]:
for col in ff5.columns:
    s = ff5[col]
    first, last = s.first_valid_index(), s.last_valid_index()
    gaps = s.loc[first:last].isna().sum()
    print(f'{col:4}  {first.date()} -> {last.date()}  | NaN inside: {gaps}')

MKT   2011-12-31 -> 2026-07-31  | NaN inside: 0
SMB   2016-07-31 -> 2026-06-30  | NaN inside: 0
HML   2015-07-31 -> 2026-06-30  | NaN inside: 0
RMW   2015-07-31 -> 2026-06-30  | NaN inside: 0
CMA   2016-07-31 -> 2026-06-30  | NaN inside: 0
Rf    2011-12-31 -> 2026-07-31  | NaN inside: 0


NaN values are only at the edges. Checked!

In [94]:
ff5[80:90]

,MKT,SMB,HML,RMW,CMA,Rf
date,,,,,,
2017-10-31,0.072907,0.000677,0.069929,-0.067509,0.067494,0.005083
2017-11-30,-0.004410,0.042103,-0.056637,-0.007967,-0.013777,0.005073
2017-12-31,0.032564,0.021372,-0.003237,-0.004970,0.008757,0.005090
2018-01-31,-0.004425,-0.013034,0.025481,-0.041662,0.002902,0.005138
2018-02-28,-0.034234,0.014162,-0.027857,0.012994,-0.023509,0.005280
2018-03-31,-0.051660,-0.004340,-0.084952,-0.003166,-0.012905,0.005315
2018-04-30,0.058524,0.009275,-0.020622,0.021091,-0.025899,0.005193
2018-05-31,-0.033362,-0.007646,-0.034530,0.013345,-0.014286,0.005107
2018-06-30,-0.066983,-0.007616,0.030813,0.018643,-0.024157,0.005254


***What one SMB value means?***

SMB in month t = how much the small-stock side returned minus the big-stock side, in that month.
S = (SL+SN+SH)/3 in that duration

Similarly in HML, RMW, and CMA.

***Other values***

-- Rf represent monthly risk-free returns in 91-day Treasury bills.

-- MKT = Rm - Rf, where Rm is the value-weighted return of all panel stocks (financials included), weighted by last month’s ME (me_lag)

In [95]:
# 11.2 Keep months where ALL factors exist; check no gaps

ff5 = ff5.dropna().sort_index()
ff5.index.name = 'date'

print(ff5.index.min().date(), '->', ff5.index.max().date(), '|', len(ff5), 'months')

# Every step should be exactly one month
month_id = ff5.index.year * 12 + ff5.index.month
gaps = pd.Series(month_id).diff().value_counts()

print(gaps)                                   # only 1.0 expected

2016-07-31 -> 2026-06-30 | 120 months
date
1.0    119
Name: count, dtype: int64


In [102]:
# 11.3 Save for Phase 3

ff5.reset_index().to_csv(PROCESSED_DIR / 'phase_2_ff5_factors.csv', index=False)
print('Saved: phase_2_ff5_factors.csv')

Saved: phase_2_ff5_factors.csv


In [97]:
# 11.4 Summary table (the headline result of Phase 2)

cols = ['MKT', 'SMB', 'HML', 'RMW', 'CMA']
n = len(ff5)

summary = pd.DataFrame({
    'mean_monthly_%': ff5[cols].mean() * 100,
    'ann_mean_%': ff5[cols].mean() * 12 * 100,
    'ann_vol_%': ff5[cols].std() * np.sqrt(12) * 100,
    't_stat': ff5[cols].mean() / (ff5[cols].std() / np.sqrt(n)),
})
print(summary.round(2))

print(ff5[cols].corr().round(2))

     mean_monthly_%  ann_mean_%  ann_vol_%  t_stat
MKT            0.45        5.36      17.58    0.96
SMB            1.30       15.58      12.25    4.02
HML            0.84       10.09      14.45    2.21
RMW           -0.24       -2.84      11.46   -0.78
CMA            0.08        1.02       9.07    0.36
      MKT   SMB   HML   RMW   CMA
MKT  1.00  0.27  0.29 -0.28 -0.15
SMB  0.27  1.00  0.30  0.05 -0.04
HML  0.29  0.30  1.00 -0.40  0.31
RMW -0.28  0.05 -0.40  1.00 -0.09
CMA -0.15 -0.04  0.31 -0.09  1.00


| Threshold | Status |
|---|---|
| \|t\| > 2 (5% level) | The long-standing convention in empirical finance, including FF-style papers |
| \|t\| > 3 | Proposed by Harvey, Liu & Zhu (2016) for **new** factors. So many factors have been tested that some pass t > 2 by luck (data mining) |

## Phase 2 Results: FF5 Factor Premia (Jul 2016 – Jun 2026)

| Factor | Mean (ann. %) | Vol (ann. %) | t-stat | Significant? |
|---|---:|---:|---:|:---:|
| MKT | 5.54 | 17.52 | 1.00 | ❌ |
| SMB | 15.23 | 12.25 | 3.95 | ✅ |
| HML | 9.91 | 14.40 | 2.19 | ✅ |
| RMW | −2.46 | 11.47 | −0.68 | ❌ |
| CMA | 0.96 | 9.03 | 0.34 | ❌ |

### Interpretation

- **Sample:** Jul 2016 – Jun 2026 (120 months). That's short for factor inference: a factor
  with about 12% volatility needs roughly 7–8% a year to reach t = 2.

- **MKT (5.5%, t = 1.0):** Positive but not significant. High volatility (17.5%) over only
  10 years makes significance hard. It's also pulled down by price-only returns (no dividends,
  about 1–1.5% a year) and a relatively high Indian Rf (about 6%). The COVID crash and the
  2024–25 correction sit inside the window.

- **SMB (15.2%, t = 3.95):** The strongest result, matching the mid- and small-cap rally of
  2020–24. *Treat it with caution:* a current-constituent universe includes companies that were
  small and grew into the Nifty 500 (survivorship and look-ahead), which inflates SMB.
  "Small" here also means mid-cap.

- **HML (9.9%, t = 2.19):** Significant at 5%. Consistent with the value/PSU re-rating of
  2021–24 (PSU banks, energy, metals). It's driven by one regime, so it may not hold over
  longer periods.

- **RMW (−2.5%, t = −0.68):** Not different from zero. High-profitability "quality" stocks
  (e.g., FMCG) lagged as cyclicals and PSUs rallied. Financials are excluded, which narrows
  the sample.

- **CMA (1.0%, t = 0.34):** Essentially zero, so there's no investment premium in this window.
  Check its correlation with HML: FF (2015) found the two overlap strongly.

### Overall

- Only **SMB** and **HML** pass \|t\| > 2. Low t-stats elsewhere reflect the short sample,
  not necessarily that the premium is absent.

### Technical caveats

- The t-stats assume independent months, so **Newey-West** standard errors will be used in Phase 3.
- **Survivorship bias** is the main threat to SMB and HML.
- Whether factors are **redundant** (e.g., HML given RMW and CMA) gets tested in Phase 3.